# 02 — Intent parsing and eligibility KG

Age and explicitly requested term limits are hard gates. Missing age yields `unknown`, never eligible. The graph preserves the catalogue's claimed verification text but marks every policy unverified.

In [ ]:
import importlib.util, json, random, subprocess, sys
from pathlib import Path
_REQUIRED = {"pandas": "pandas", "pyarrow": "pyarrow", "networkx": "networkx"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
import pandas as pd
import networkx as nx
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

In [ ]:
import re

policies_df = pd.read_parquet(ARTIFACTS / "policies.parquet")
queries_df = pd.read_csv(ARTIFACTS / "queries.csv")

GOAL_KEYWORDS = {
    "Life Cover": ("term", "life cover", "protection", "death benefit", "sum assured"),
    "Child Education": ("child", "daughter", "son", "education", "school", "college", "marriage"),
    "Retirement": ("retirement", "pension", "annuity", "retire", "old age"),
    "Savings": ("savings", "endowment", "guaranteed", "money back", "bachat", "maturity"),
    "Wealth Creation": ("ulip", "wealth", "market", "equity", "high returns", "growth"),
    "Health Protection": ("cancer", "health", "hospital", "mediclaim", "illness", "medical", "icu"),
    "Income": ("regular income", "monthly income", "annual income", "survival benefit", "cash flow"),
}


def parse_intent(text):
    text = text if isinstance(text, str) else ""
    normalized = text.lower()
    child_matches = []
    for pattern in (
        r"(?P<age>\d{1,2})\s*(?:year|yr)s?\s*old\s*(?:daughter|son|child|kid|baby)\b",
        r"(?:daughter|son|child|kid)\s*(?:is|aged|of)\s*(?P<age>\d{1,2})\b",
    ):
        child_matches.extend(re.finditer(pattern, normalized))
    child_age = next((int(m.group("age")) for m in child_matches if int(m.group("age")) <= 25), None)

    customer_age = None
    customer_pattern = r"\b(?:i am|i'm|my age is|age|aged|at age|senior citizen aged|turning|turned)\s*(?:a\s*)?(\d{1,3})\b"
    for match in re.finditer(customer_pattern, normalized):
        start, end = match.span(1)
        if any(start < child.end("age") and end > child.start("age") for child in child_matches):
            continue
        age = int(match.group(1))
        if 16 <= age <= 105:
            customer_age = age
            break

    term_match = re.search(
        r"\b(\d{1,2})\s*(?:years?|yrs?)\s*(?:policy\s*)?term\b|\bterm\s*(?:of\s*)?(\d{1,2})\s*(?:years?|yrs?)?\b",
        normalized,
    )
    term = int(next(group for group in term_match.groups() if group is not None)) if term_match else None
    budget_match = re.search(
        r"\b(?:under|below|up to|budget(?: of)?|premium of)?\s*(?:₹|rs\.?\s*)?(\d[\d,]*(?:\.\d+)?)\s*(lakhs?|lacs?|rupees?|rs\.?|k)\b",
        normalized,
    )
    budget = None if not budget_match else {"amount": budget_match.group(1).replace(",", ""), "unit": budget_match.group(2).lower()}

    goals = [goal for goal, keywords in GOAL_KEYWORDS.items() if any(re.search(rf"\b{re.escape(keyword)}\b", normalized) for keyword in keywords)]
    risk = next((value for value, keywords in (("High", ("high risk", "equity", "aggressive", "market linked", "ulip")), ("Moderate", ("moderate risk", "balanced", "medium risk")), ("Low", ("safe", "low risk", "risk free", "guaranteed", "no risk", "zero risk", "secure"))) if any(keyword in normalized for keyword in keywords)), None)
    premium = next((value for value, keywords in (("Low", ("cheap", "low premium", "affordable", "budget", "small premium", "micro")), ("Moderate", ("moderate premium", "reasonable premium", "regular premium")), ("High", ("single premium", "lump sum", "high premium", "hni"))) if any(keyword in normalized for keyword in keywords)), None)
    contradictions = []
    if any(term in normalized for term in ("high returns", "market returns", "wealth")) and any(term in normalized for term in ("zero risk", "no risk", "risk free", "risk-free")):
        contradictions.append("High market returns conflict with a zero-risk requirement.")
    confidence = sum(value is not None for value in (customer_age, goals[0] if goals else None, risk, premium)) / 4
    return {
        "customer_age": customer_age,
        "child_age": child_age,
        "term": term,
        "budget": budget,
        "all_goals": goals,
        "primary_goal": goals[0] if goals else None,
        "risk_tolerance": risk,
        "premium_level": premium,
        "contradictions": contradictions,
        "is_ambiguous": not any((customer_age is not None, goals, risk, premium, budget)),
        "confidence": round(confidence, 2),
    }


def eligibility(policy, intent):
    age = intent.get("customer_age")
    term = intent.get("term")
    reasons = []
    if age is not None and not (int(policy["min_age"]) <= age <= int(policy["max_age"])):
        reasons.append("entry age outside catalogue limits")
    if term is not None and not (int(policy["min_term"]) <= term <= int(policy["max_term"])):
        reasons.append("requested policy term outside catalogue limits")
    if reasons:
        return {"status": "ineligible", "reasons": reasons}
    if age is None:
        return {"status": "unknown", "reasons": ["customer age not provided"]}
    return {"status": "eligible", "reasons": []}


def recommend(query, policies=policies_df):
    intent = parse_intent(query)
    states = {row["policy_id"]: eligibility(row, intent) for row in policies.to_dict(orient="records")}
    eligible_ids = sorted(policy_id for policy_id, state in states.items() if state["status"] == "eligible")
    return {
        "status": "SUCCESS" if eligible_ids else "NO_ELIGIBLE_POLICIES",
        "query": query,
        "intent": intent,
        "eligibility": states,
        "recommendations": eligible_ids,
    }


def build_kg(policies):
    graph = nx.DiGraph()
    for row in policies.to_dict(orient="records"):
        pid = str(row["policy_id"])
        graph.add_node(pid, type="Policy", policy_name=str(row["policy_name"]), min_age=int(row["min_age"]), max_age=int(row["max_age"]), min_term=int(row["min_term"]), max_term=int(row["max_term"]), verified=False, claimed_verification=str(row["claimed_verification"]), source_url=str(row["source_url"]))
        for attribute, value, relation in (
            ("Goal", row["financial_goal"], "HAS_GOAL"),
            ("Category", row["category"], "IN_CATEGORY"),
            ("Risk", row["risk_tolerance_inferred"], "SUITS_RISK"),
            ("Premium", row["premium_level_inferred"], "HAS_PREMIUM"),
        ):
            node_id = f"{attribute}:{value}"
            graph.add_node(node_id, type=attribute, value=str(value))
            graph.add_edge(pid, node_id, relation=relation)
    return graph

kg = build_kg(policies_df)
nx.write_graphml(kg, ARTIFACTS / "kg.graphml")
intent_records = [{"query_id": row.query_id, "raw_query": row.raw_query, "intent": parse_intent(row.raw_query)} for row in queries_df.itertuples(index=False)]
(ARTIFACTS / "intents.json").write_text(json.dumps(intent_records, indent=2), encoding="utf-8")
print(f"Wrote KG with {kg.number_of_nodes()} nodes and {kg.number_of_edges()} edges; intents are query-derived and synthetic-only.")

## Assertions

In [ ]:
# Exact boundaries pass; a single year outside either statutory interval fails.
policy = policies_df.iloc[0].to_dict()
for age in (int(policy["min_age"]), int(policy["max_age"])):
    assert eligibility(policy, {"customer_age": age, "term": None})["status"] == "eligible"
outside_age = int(policy["min_age"]) - 1 if int(policy["min_age"]) > 0 else int(policy["max_age"]) + 1
assert eligibility(policy, {"customer_age": outside_age, "term": None})["status"] == "ineligible"
for term in (int(policy["min_term"]), int(policy["max_term"])):
    assert eligibility(policy, {"customer_age": int(policy["min_age"]), "term": term})["status"] == "eligible"
outside_term = int(policy["min_term"]) - 1
assert eligibility(policy, {"customer_age": int(policy["min_age"]), "term": outside_term})["status"] == "ineligible"
assert eligibility(policy, {"customer_age": None, "term": None})["status"] == "unknown"

parent = parse_intent("I am 36 with a 3 year old son")
assert (parent["customer_age"], parent["child_age"]) == (36, 3)
assert parse_intent("5 Lakhs") ["customer_age"] is None
term_only = parse_intent("I need a 30 years term")
assert term_only["customer_age"] is None and term_only["term"] == 30
assert parse_intent("I want high market returns with zero risk")["contradictions"]
empty = recommend("")
assert empty["intent"]["confidence"] == 0.0
assert empty["status"] == "NO_ELIGIBLE_POLICIES" and not empty["recommendations"]
old = recommend("I am 95 years old and want a policy")
assert old["status"] == "NO_ELIGIBLE_POLICIES" and not old["recommendations"]
assert all(state["status"] != "eligible" for state in old["eligibility"].values())
assert recommend("I am 36 years old looking for life cover")["recommendations"] == recommend("I am 36 years old looking for life cover")["recommendations"]
loaded_graph = nx.read_graphml(ARTIFACTS / "kg.graphml")
assert sum(node_data.get("type") == "Policy" for _, node_data in loaded_graph.nodes(data=True)) == len(policies_df)
assert all(not node_data.get("verified", True) for _, node_data in loaded_graph.nodes(data=True) if node_data.get("type") == "Policy")
print("Notebook 02 checks passed: age/term gates, unknown status, parser, strict refusal, determinism, and GraphML provenance.")